# 1. Data audit and feature definition

## Small idea: understand rows before transforming columns

A preprocessing plan begins with the prediction unit, target, feature availability,
data types, missingness, duplicates, ranges, and sampling structure. Cleaning without
this contract can silently change the research question.

**Learning goals**

- define the unit of analysis and prediction time;
- create a compact quality report;
- identify IDs, audit-only variables, and target-derived leakage;
- flag duplicates and unusual values without deleting them blindly.

In [ ]:
import hashlib
import numpy as np
import pandas as pd

learner_data = pd.DataFrame({
    "response_id": [f"R{i:02d}" for i in range(1, 11)],
    "learner_id": ["L01", "L01", "L02", "L03", "L04", "L05", "L06", "L07", "L08", "L08"],
    "text_raw": [
        "من فارسی را دوست دارم", "من فارسی را دوست دارم!", "من كتاب می خوانم",
        "یادگیریِ زبان جالب است", "زبان فارسی سخت نیست", "سلام... فارسی خوبه 😊",
        "من به دانشگاه می‌روم", "این تمرین خیلییی سخت بود", "کتاب جدید خریدم", "کتاب جدید خریدم",
    ],
    "task_type": ["free", "free", "picture", "free", "picture", "social", "free", "social", "picture", "picture"],
    "proficiency": ["B1", "B1", "A2", "B2", "B1", "A2", "B2", "A2", "B1", "B1"],
    "age": [24, 24, 29, 34, np.nan, 21, 27, 22, 1_200, 26],
    "first_language": ["Arabic", "Arabic", "Arabic", "Kurdish", "Arabic", "Turkish", "Arabic", "Arabic", "Kurdish", "Kurdish"],
    "annotated_errors": [2, 1, 5, 1, 3, 7, 1, 6, 2, 2],
    "needs_review": [0, 0, 1, 0, 0, 1, 0, 1, 0, 0],
})
learner_data

## 1. Write the data contract

Here the unit is **one written response**, and the target is whether the response needs
manual review. `annotated_errors` is excluded because it was used to help create the
target and would not be available before annotation.

In [ ]:
contract = {
    "unit": "one written response",
    "target": "needs_review",
    "features_available_at_prediction": ["text_raw", "task_type", "proficiency", "age"],
    "identifiers": ["response_id", "learner_id"],
    "audit_only": ["first_language"],
    "target_derived": ["annotated_errors"],
}

feature_columns = contract["features_available_at_prediction"]
forbidden = contract["identifiers"] + contract["audit_only"] + contract["target_derived"]
assert set(feature_columns).isdisjoint(forbidden)
contract

## 2. Inspect shape, types, missingness, and uniqueness

In [ ]:
quality_report = pd.DataFrame({
    "dtype": learner_data.dtypes.astype(str),
    "missing_n": learner_data.isna().sum(),
    "missing_pct": learner_data.isna().mean().mul(100).round(1),
    "unique_n": learner_data.nunique(dropna=False),
})
quality_report

In [ ]:
print("Rows, columns:", learner_data.shape)
print("Duplicate response IDs:", learner_data["response_id"].duplicated().sum())
print("Target distribution:\n", learner_data["needs_review"].value_counts(dropna=False))
print("Responses per learner:\n", learner_data["learner_id"].value_counts().head())

## 3. Preserve raw text and detect exact duplicates

Never overwrite the only copy of the raw text. Create additional views later. Exact
duplicate detection can use normalized text or a stable hash, but duplicate rows should
be inspected in context: repeated responses may be errors, reposts, templates, or valid
repeated observations.

In [ ]:
learner_data["text_hash"] = learner_data["text_raw"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
)

duplicate_mask = learner_data.duplicated("text_hash", keep=False)
learner_data.loc[duplicate_mask, ["response_id", "learner_id", "text_raw", "text_hash"]]

## 4. Flag implausible or unusual values

A value can be statistically unusual yet scientifically valid. Flag first, inspect the
source, document the decision, and avoid using the target to decide which rows to remove.

In [ ]:
age_bounds = learner_data["age"].between(15, 90) | learner_data["age"].isna()
age_issues = learner_data.loc[~age_bounds, ["response_id", "age"]]
age_issues

In [ ]:
text_lengths = learner_data["text_raw"].str.len()
q1, q3 = text_lengths.quantile([0.25, 0.75])
iqr = q3 - q1
unusual_length = ~text_lengths.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

learner_data.loc[unusual_length, ["response_id", "text_raw"]]

## 5. Audit-only variables still matter

First language, gender, nationality, or institution may be important for describing the
population and checking performance slices. They should not automatically become model
features. Their use requires a scientific, ethical, and deployment justification.

In [ ]:
representation = pd.crosstab(
    learner_data["first_language"],
    learner_data["proficiency"],
    margins=True,
)
representation

## Audit checklist

- What does one row represent?
- When is the prediction made, and which columns exist at that time?
- Is the target directly or indirectly encoded in a feature?
- Do participants, sources, dates, or duplicates create dependence between rows?
- Which variables are identifiers or audit-only?
- Are missing and unusual values plausible, structural, or data-entry errors?

## Tiny checkpoint

Create a second contract for predicting `proficiency`. Which current columns would become
forbidden because they are measured after or derived from proficiency assessment?